In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [3]:
DATA_PATH = "S06-hw-dataset-02.csv"  # относительный путь

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))


Shape: (18000, 39)
Columns: ['id', 'f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31', 'f32', 'f33', 'f34', 'f35', 'x_int_1', 'x_int_2', 'target']


In [4]:
df.head(10)
df.info()
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 39 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       18000 non-null  int64  
 1   f01      18000 non-null  float64
 2   f02      18000 non-null  float64
 3   f03      18000 non-null  float64
 4   f04      18000 non-null  float64
 5   f05      18000 non-null  float64
 6   f06      18000 non-null  float64
 7   f07      18000 non-null  float64
 8   f08      18000 non-null  float64
 9   f09      18000 non-null  float64
 10  f10      18000 non-null  float64
 11  f11      18000 non-null  float64
 12  f12      18000 non-null  float64
 13  f13      18000 non-null  float64
 14  f14      18000 non-null  float64
 15  f15      18000 non-null  float64
 16  f16      18000 non-null  float64
 17  f17      18000 non-null  float64
 18  f18      18000 non-null  float64
 19  f19      18000 non-null  float64
 20  f20      18000 non-null  float64
 21  f21      180

,count,mean,std,min,25%,50%,75%,max
id,18000.0,9000.500000,5196.296758,1.000000e+00,4500.750000,9000.500000,13500.250000,18000.000000
f01,18000.0,-0.418555,2.178005,-1.001470e+01,-1.866134,-0.465100,0.966393,9.589975
f02,18000.0,0.614251,3.926778,-1.551032e+01,-2.048192,0.600291,3.229850,15.417329
f03,18000.0,0.004559,1.000134,-4.031762e+00,-0.673127,0.003581,0.671390,3.817025
f04,18000.0,0.059000,5.713672,-2.366326e+01,-3.544964,0.072826,3.689490,26.815691
f05,18000.0,0.405086,2.497581,-1.228931e+01,-1.153000,0.485625,2.075739,10.665184
f06,18000.0,0.012123,0.987226,-3.741536e+00,-0.653090,0.018765,0.689304,3.528280
f07,18000.0,-0.283473,2.193891,-9.591425e+00,-1.743214,-0.251263,1.195481,7.794627
f08,18000.0,-0.266880,2.081431,-8.293319e+00,-1.688121,-0.302463,1.109589,8.892834
f09,18000.0,0.255107,2.225776,-1.365574e+01,-1.177480,0.350739,1.764113,8.699629


In [5]:
if "target" not in df.columns:
    raise ValueError("В датасете не найден столбец 'target'.")

target_counts = df["target"].value_counts(dropna=False)
target_share = df["target"].value_counts(normalize=True, dropna=False)

pd.DataFrame({"count": target_counts, "share": target_share})


,count,share
target,,
0,13273,0.737389
1,4727,0.262611


In [6]:
# Пропуски
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Колонки с пропусками:", len(missing))
missing.to_frame("missing_count")
# Типы
df.dtypes.to_frame("dtype").T


Колонки с пропусками: 0


,id,f01,f02,f03,f04,f05,f06,f07,f08,f09,...,f29,f30,f31,f32,f33,f34,f35,x_int_1,x_int_2,target
dtype,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,...,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64


In [7]:
y = df["target"].copy()

drop_cols = ["target"]
if "id" in df.columns:
    drop_cols.append("id")

X = df.drop(columns=drop_cols).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("id in X?", "id" in X.columns)
print("target in X?", "target" in X.columns)

X.head(5)


X shape: (18000, 37)
y shape: (18000,)
id in X? False
target in X? False


,f01,f02,f03,f04,f05,f06,f07,f08,f09,f10,...,f28,f29,f30,f31,f32,f33,f34,f35,x_int_1,x_int_2
0,-0.149235,-2.826966,-0.522901,-4.198449,1.364943,0.815043,-1.195518,-1.932232,2.396353,1.121683,...,0.293322,-0.159323,0.448015,0.572745,0.149916,0.878392,-0.679733,1.412751,0.421883,9.217167
1,-1.966180,-4.877542,0.268367,-9.607791,0.097149,1.347185,-3.872575,-0.395117,1.710068,-0.298809,...,1.924549,-0.389212,1.383794,0.169876,0.043969,-0.963545,1.006643,-2.488690,9.590124,24.772826
2,-0.555964,-0.999920,0.209673,-14.119498,-1.808950,-0.006222,-4.651108,0.911944,-0.289037,-0.092692,...,0.792870,-1.383970,3.044321,-0.182864,1.425649,-8.418598,-4.629754,-0.439798,0.555919,41.800517
3,-2.049199,-5.600713,-1.664677,-6.263893,-5.224455,0.848351,1.407210,-0.542080,0.119102,0.681733,...,-0.732601,-2.713080,2.762637,-0.520796,-0.142455,1.668338,2.292810,-10.744916,11.476977,65.315860
4,-0.220556,4.889479,-2.235840,6.450046,0.774389,-2.382625,2.584816,4.211856,-0.317889,-1.275888,...,1.948262,-1.302872,2.478862,1.528610,1.098131,3.547087,2.517757,-9.364106,-1.078404,93.017870


In [9]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.25
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)

# Проверим доли классов (должны быть почти одинаковые)
train_share = y_train.value_counts(normalize=True).sort_index()
test_share  = y_test.value_counts(normalize=True).sort_index()

display(pd.DataFrame({"train_share": train_share, "test_share": test_share}))


Train: (13500, 37) (13500,)
Test:  (4500, 37) (4500,)


,train_share,test_share
target,,
0,0.737407,0.737333
1,0.262593,0.262667


Почему важны фиксированный seed и стратификация:

Фиксированный random_state (seed) делает эксперимент воспроизводимым: при повторном запуске получится тот же сплит и можно будет честно сравнивать модели/гиперпараметры. Без этого метрики могут “прыгать” просто из-за другого разбиения.

stratify=y сохраняет примерно одинаковые доли классов в train и test. Это особенно важно при дисбалансе: иначе можно случайно получить тест, где почти нет одного из классов, и метрики станут некорректными/нестабильными.

In [10]:
def evaluate_on_test(name, model, X_test, y_test):
    """
    model: уже обученная модель (fit уже был сделан)
    """
    y_pred = model.predict(X_test)

    # roc_auc считаем только если есть predict_proba (и бинарная классификация)
    roc_auc = np.nan
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        if proba.shape[1] == 2:  # бинарный случай
            roc_auc = roc_auc_score(y_test, proba[:, 1])

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"\n=== {name} ===")
    print("Accuracy:", acc)
    print("F1:", f1)
    if not np.isnan(roc_auc):
        print("ROC-AUC:", roc_auc)
    print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

    return {
        "model": name,
        "accuracy": acc,
        "f1": f1,
        "roc_auc": roc_auc
    }

results = []


In [11]:
# 1-ы Baseline: DummyClassifier
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)

results.append(evaluate_on_test("DummyClassifier(most_frequent)", dummy, X_test, y_test))



=== DummyClassifier(most_frequent) ===
Accuracy: 0.7373333333333333
F1: 0.0
ROC-AUC: 0.5

Confusion matrix:
 [[3318    0]
 [1182    0]]

Classification report:
               precision    recall  f1-score   support

           0     0.7373    1.0000    0.8488      3318
           1     0.0000    0.0000    0.0000      1182

    accuracy                         0.7373      4500
   macro avg     0.3687    0.5000    0.4244      4500
weighted avg     0.5437    0.7373    0.6259      4500



C:\Users\sveti\PycharmProjects\DPO_II\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sveti\PycharmProjects\DPO_II\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sveti\PycharmProjects\DPO_II\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [12]:
# 2-й Baseline: LogisticRegression
logreg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, random_state=42))
])

logreg.fit(X_train, y_train)

results.append(evaluate_on_test("LogReg(Scaler+LR)", logreg, X_test, y_test))



=== LogReg(Scaler+LR) ===
Accuracy: 0.8162222222222222
F1: 0.5717244950802693
ROC-AUC: 0.8008904412072182

Confusion matrix:
 [[3121  197]
 [ 630  552]]

Classification report:
               precision    recall  f1-score   support

           0     0.8320    0.9406    0.8830      3318
           1     0.7370    0.4670    0.5717      1182

    accuracy                         0.8162      4500
   macro avg     0.7845    0.7038    0.7274      4500
weighted avg     0.8071    0.8162    0.8012      4500



DummyClassifier показывает “наивный уровень” — насколько легко получить метрику без реального обучения.

LogisticRegression — сильный линейный baseline: если ансамбли выигрывают у неё, значит в данных есть важная нелинейность/взаимодействия.

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Несколько метрик в GridSearch, а refit — по roc_auc (можешь сменить на "f1")
scoring = {
    "roc_auc": "roc_auc",
    "f1": "f1",
    "accuracy": "accuracy"
}
REFIT_METRIC = "roc_auc"


In [14]:
def evaluate_on_test(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    roc_auc = np.nan
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        if proba.shape[1] == 2:
            roc_auc = roc_auc_score(y_test, proba[:, 1])

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"\n=== {name} ===")
    print("Accuracy:", acc)
    print("F1:", f1)
    if not np.isnan(roc_auc):
        print("ROC-AUC:", roc_auc)
    print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

    return {"model": name, "accuracy": acc, "f1": f1, "roc_auc": roc_auc}


In [15]:
depth_grid = {"max_depth": [2, 3, 5, 8, 12, None]}
results = []


In [16]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=RANDOM_STATE,
    min_samples_leaf=5   # фиксируем для стабильности (можешь поставить 1 или 10)
)

dt_search = GridSearchCV(
    dt, depth_grid,
    scoring=scoring,
    refit=REFIT_METRIC,
    cv=cv,
    n_jobs=2,
    verbose=0
)

dt_search.fit(X_train, y_train)
print("DT best params:", dt_search.best_params_, "best CV:", dt_search.best_score_)

best_dt = dt_search.best_estimator_
results.append(evaluate_on_test("DecisionTree (tune max_depth)", best_dt, X_test, y_test))


DT best params: {'max_depth': 8} best CV: 0.8155220353367303

=== DecisionTree (tune max_depth) ===
Accuracy: 0.8244444444444444
F1: 0.6131243878550441
ROC-AUC: 0.8314200907932836

Confusion matrix:
 [[3084  234]
 [ 556  626]]

Classification report:
               precision    recall  f1-score   support

           0     0.8473    0.9295    0.8865      3318
           1     0.7279    0.5296    0.6131      1182

    accuracy                         0.8244      4500
   macro avg     0.7876    0.7295    0.7498      4500
weighted avg     0.8159    0.8244    0.8147      4500



In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,          # фиксируем
    random_state=RANDOM_STATE,
    n_jobs=1,                  # чтобы не вешать систему
    max_features="sqrt",       # фиксируем
    min_samples_leaf=1         # фиксируем
)

rf_search = GridSearchCV(
    rf, depth_grid,
    scoring=scoring,
    refit=REFIT_METRIC,
    cv=cv,
    n_jobs=2,
    verbose=0
)

rf_search.fit(X_train, y_train)
print("RF best params:", rf_search.best_params_, "best CV:", rf_search.best_score_)

best_rf = rf_search.best_estimator_
results.append(evaluate_on_test("RandomForest (tune max_depth)", best_rf, X_test, y_test))


RF best params: {'max_depth': None} best CV: 0.9272940503067755

=== RandomForest (tune max_depth) ===
Accuracy: 0.8913333333333333
F1: 0.7565953210552514
ROC-AUC: 0.9293588068567185

Confusion matrix:
 [[3251   67]
 [ 422  760]]

Classification report:
               precision    recall  f1-score   support

           0     0.8851    0.9798    0.9301      3318
           1     0.9190    0.6430    0.7566      1182

    accuracy                         0.8913      4500
   macro avg     0.9020    0.8114    0.8433      4500
weighted avg     0.8940    0.8913    0.8845      4500



In [18]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
    n_estimators=200,     # фиксируем
    learning_rate=0.05,   # фиксируем
    min_samples_leaf=5    # фиксируем
)

gb_search = GridSearchCV(
    gb, depth_grid,
    scoring=scoring,
    refit=REFIT_METRIC,
    cv=cv,
    n_jobs=2,             # НЕ -1
    verbose=0
)

gb_search.fit(X_train, y_train)
print("GB best params:", gb_search.best_params_, "best CV:", gb_search.best_score_)

best_gb = gb_search.best_estimator_
results.append(evaluate_on_test("GradientBoosting (tune max_depth)", best_gb, X_test, y_test))


GB best params: {'max_depth': 8} best CV: 0.9306820041385105

=== GradientBoosting (tune max_depth) ===
Accuracy: 0.9086666666666666
F1: 0.8091035764050163
ROC-AUC: 0.9327416267112983

Confusion matrix:
 [[3218  100]
 [ 311  871]]

Classification report:
               precision    recall  f1-score   support

           0     0.9119    0.9699    0.9400      3318
           1     0.8970    0.7369    0.8091      1182

    accuracy                         0.9087      4500
   macro avg     0.9044    0.8534    0.8745      4500
weighted avg     0.9080    0.9087    0.9056      4500



In [19]:
# ---- единый список моделей для metrics_test.* (5 обязательных) ----
models_for_metrics = {
    "DummyClassifier": dummy,
    "LogisticRegression": logreg,
    "DecisionTree": best_dt,
    "RandomForest": best_rf,
    "Boosting": best_gb,
}

metrics_results = []
for name, model in models_for_metrics.items():
    metrics_results.append(evaluate_on_test(name, model, X_test, y_test))

metrics_test_df = pd.DataFrame(metrics_results).sort_values(
    by=["roc_auc", "f1", "accuracy"], ascending=False
).reset_index(drop=True)

metrics_test_df



=== DummyClassifier ===
Accuracy: 0.7373333333333333
F1: 0.0
ROC-AUC: 0.5

Confusion matrix:
 [[3318    0]
 [1182    0]]

Classification report:
               precision    recall  f1-score   support

           0     0.7373    1.0000    0.8488      3318
           1     0.0000    0.0000    0.0000      1182

    accuracy                         0.7373      4500
   macro avg     0.3687    0.5000    0.4244      4500
weighted avg     0.5437    0.7373    0.6259      4500


=== LogisticRegression ===
Accuracy: 0.8162222222222222
F1: 0.5717244950802693
ROC-AUC: 0.8008904412072182

Confusion matrix:
 [[3121  197]
 [ 630  552]]

Classification report:
               precision    recall  f1-score   support

           0     0.8320    0.9406    0.8830      3318
           1     0.7370    0.4670    0.5717      1182

    accuracy                         0.8162      4500
   macro avg     0.7845    0.7038    0.7274      4500
weighted avg     0.8071    0.8162    0.8012      4500


=== DecisionTree =

C:\Users\sveti\PycharmProjects\DPO_II\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sveti\PycharmProjects\DPO_II\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sveti\PycharmProjects\DPO_II\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital


=== RandomForest ===
Accuracy: 0.8913333333333333
F1: 0.7565953210552514
ROC-AUC: 0.9293588068567185

Confusion matrix:
 [[3251   67]
 [ 422  760]]

Classification report:
               precision    recall  f1-score   support

           0     0.8851    0.9798    0.9301      3318
           1     0.9190    0.6430    0.7566      1182

    accuracy                         0.8913      4500
   macro avg     0.9020    0.8114    0.8433      4500
weighted avg     0.8940    0.8913    0.8845      4500


=== Boosting ===
Accuracy: 0.9086666666666666
F1: 0.8091035764050163
ROC-AUC: 0.9327416267112983

Confusion matrix:
 [[3218  100]
 [ 311  871]]

Classification report:
               precision    recall  f1-score   support

           0     0.9119    0.9699    0.9400      3318
           1     0.8970    0.7369    0.8091      1182

    accuracy                         0.9087      4500
   macro avg     0.9044    0.8534    0.8745      4500
weighted avg     0.9080    0.9087    0.9056      4500



,model,accuracy,f1,roc_auc
0,Boosting,0.908667,0.809104,0.932742
1,RandomForest,0.891333,0.756595,0.929359
2,DecisionTree,0.824444,0.613124,0.831420
3,LogisticRegression,0.816222,0.571724,0.800890
4,DummyClassifier,0.737333,0.000000,0.500000


In [20]:
# Сравнение моделей
results_df = pd.DataFrame(results).sort_values(by=["roc_auc", "f1", "accuracy"], ascending=False)
results_df


,model,accuracy,f1,roc_auc
2,GradientBoosting (tune max_depth),0.908667,0.809104,0.932742
1,RandomForest (tune max_depth),0.891333,0.756595,0.929359
0,DecisionTree (tune max_depth),0.824444,0.613124,0.831420


In [21]:
# ====== Метрики + графики (ROC, Confusion Matrix, PR)
import re
from pathlib import Path

FIG_DIR = Path("artifacts") / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def _safe_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9_\-]+", "_", s)
    return s.strip("_")

def _save_fig(filename: str, dpi: int = 150):
    path = FIG_DIR / filename
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("Saved:", path)

def evaluate_binary_with_plots(model, X_test, y_test, name: str, save_pr: bool = False):
    """
    Для бинарной классификации:
    - считает accuracy, f1, ROC-AUC (если есть predict_proba)
    - сохраняет confusion matrix и ROC-кривую
    - опционально сохраняет PR-кривую (полезно при дисбалансе)
    """
    # Предсказания класса
    y_pred = model.predict(X_test)

    # Метрики
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Вероятности для ROC-AUC и кривых
    roc_auc = np.nan
    y_score = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        if proba.shape[1] == 2:
            y_score = proba[:, 1]
            roc_auc = roc_auc_score(y_test, y_score)

    print(f"\n=== {name} ===")
    print("Accuracy:", acc)
    print("F1:", f1)
    if not np.isnan(roc_auc):
        print("ROC-AUC:", roc_auc)
    else:
        print("ROC-AUC: не посчитан (у модели нет predict_proba)")

    tag = _safe_name(name)

    # --- Confusion matrix ---
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax)
    ax.set_title(f"Confusion Matrix — {name}")
    _save_fig(f"{tag}_confusion_matrix.png")

    # --- ROC curve ---
    if y_score is not None:
        fig, ax = plt.subplots()
        RocCurveDisplay.from_predictions(y_test, y_score, ax=ax)
        ax.set_title(f"ROC Curve — {name}")
        _save_fig(f"{tag}_roc_curve.png")

        # --- PR curve (optional) ---
        if save_pr:
            fig, ax = plt.subplots()
            PrecisionRecallDisplay.from_predictions(y_test, y_score, ax=ax)
            ax.set_title(f"PR Curve — {name}")
            _save_fig(f"{tag}_pr_curve.png")

    return {"model": name, "accuracy": acc, "f1": f1, "roc_auc": roc_auc}

# ====== Использование (графики) ======
# best_dt, best_rf, best_gb должны быть уже обучены (GridSearchCV.fit был сделан)

plot_results = []
plot_results.append(evaluate_binary_with_plots(best_dt, X_test, y_test, name="DecisionTree", save_pr=False))
plot_results.append(evaluate_binary_with_plots(best_rf, X_test, y_test, name="RandomForest", save_pr=False))
plot_results.append(evaluate_binary_with_plots(best_gb, X_test, y_test, name="GradientBoosting", save_pr=False))

plot_results_df = pd.DataFrame(plot_results).sort_values(
    by=["roc_auc", "f1", "accuracy"], ascending=False
).reset_index(drop=True)

plot_results_df


=== DecisionTree ===
Accuracy: 0.8244444444444444
F1: 0.6131243878550441
ROC-AUC: 0.8314200907932836
Saved: artifacts\figures\decisiontree_confusion_matrix.png
Saved: artifacts\figures\decisiontree_roc_curve.png

=== RandomForest ===
Accuracy: 0.8913333333333333
F1: 0.7565953210552514
ROC-AUC: 0.9293588068567185
Saved: artifacts\figures\randomforest_confusion_matrix.png
Saved: artifacts\figures\randomforest_roc_curve.png

=== GradientBoosting ===
Accuracy: 0.9086666666666666
F1: 0.8091035764050163
ROC-AUC: 0.9327416267112983
Saved: artifacts\figures\gradientboosting_confusion_matrix.png
Saved: artifacts\figures\gradientboosting_roc_curve.png


,model,accuracy,f1,roc_auc
0,GradientBoosting,0.908667,0.809104,0.932742
1,RandomForest,0.891333,0.756595,0.929359
2,DecisionTree,0.824444,0.613124,0.831420


In [22]:
from sklearn.inspection import permutation_importance
import pandas as pd

# Лучшая модель по твоей таблице:
best_model = best_gb   # GradientBoosting (tune max_depth)

perm = permutation_importance(
    best_model,
    X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc"   # согласованный критерий для бинарной задачи
)

imp_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

# Top-15 (можешь заменить на 10)
imp_top15 = imp_df.head(15)
imp_top15


,feature,importance_mean,importance_std
15,f16,0.083220,0.003071
0,f01,0.030981,0.001585
6,f07,0.018180,0.001211
14,f15,0.014255,0.001050
18,f19,0.011513,0.001207
7,f08,0.011443,0.001055
22,f23,0.011247,0.001329
28,f29,0.009933,0.001009
11,f12,0.009435,0.000992
29,f30,0.008789,0.000729


Permutation importance показал, что наиболее сильное влияние на качество модели оказывает признак f16: при его перемешивании среднее падение ROC-AUC составляет примерно 0.083 (±0.003). Это заметно больше, чем у остальных признаков, поэтому f16 — ключевой фактор для разделения классов.

Далее по важности идут f01 (~0.031) и f07 (~0.018) — они тоже существенны, но их вклад уже значительно ниже, чем у f16. Остальные признаки из топа (f15, f19, f08, f23, f29, f12, f30, f18, f05, f13, f34...) дают более умеренный вклад (примерно 0.006–0.014), то есть работают скорее как “дополняющие” сигналы.

Такое распределение важностей выглядит ожидаемым для синтетического датасета со взаимодействиями и нелинейностями: бустинг обычно выделяет несколько действительно информативных признаков (здесь явно выделяется один лидер), а затем использует группу дополнительных признаков, которые улучшают качество за счёт комбинирования и пороговых эффектов.

In [24]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timezone

ART_DIR = Path("artifacts")
FIG_DIR = ART_DIR / "figures"
ART_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 1) metrics_test.json / metrics_test.csv ----------
metrics_df = metrics_test_df.copy()

# results: список записей (это ок), но корнем файла будет объект
results_records = (
    metrics_df.replace({np.nan: None})
              .to_dict(orient="records")
)

metrics_payload = {
    "meta": {
        "schema_version": 1,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(globals().get("DATA_PATH", "")),
        "test_size": float(globals().get("TEST_SIZE", 0.0)),
        "random_state": int(globals().get("RANDOM_STATE", 0)),
        "selection_criterion": ["roc_auc", "f1", "accuracy"],
        "n_models": len(results_records),
    },
    "results": results_records
}

(ART_DIR / "metrics_test.json").write_text(
    json.dumps(metrics_payload, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
metrics_df.to_csv(ART_DIR / "metrics_test.csv", index=False)

print("Saved:", (ART_DIR / "metrics_test.json").resolve())

# ---------- 2) search_summaries.json ----------
# собираем лучшие параметры и CV-score из объектов GridSearchCV, если они есть
search_summaries = {}

def add_search_summary(key, search_obj):
    if search_obj is None:
        return
    if hasattr(search_obj, "best_params_") and hasattr(search_obj, "best_score_"):
        search_summaries[key] = {
            "best_params": search_obj.best_params_,
            "best_cv_score": float(search_obj.best_score_),
            "refit_metric": globals().get("REFIT_METRIC", None)
        }

add_search_summary("DecisionTree", globals().get("dt_search", None))
add_search_summary("RandomForest", globals().get("rf_search", None))
add_search_summary("GradientBoosting", globals().get("gb_search", None))

(ART_DIR / "search_summaries.json").write_text(
    json.dumps(search_summaries, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Saved:", (ART_DIR / "search_summaries.json").resolve())

# ---------- 3) best_model.joblib + best_model_meta.json ----------
# лучшая модель по согласованному критерию (у тебя сортировка по roc_auc, потом f1, accuracy)
metrics_df_sorted = metrics_df.sort_values(by=["roc_auc", "f1", "accuracy"], ascending=False).reset_index(drop=True)
best_row = metrics_df_sorted.iloc[0].to_dict()

# Определи объект лучшей модели:
# 1) если у тебя есть best_gb / best_rf / best_dt — возьмём по названию
best_model_obj = None
model_name = str(best_row.get("model", ""))

# подхватим из известных переменных
candidates = {
    "GradientBoosting": globals().get("best_gb", None),
    "RandomForest": globals().get("best_rf", None),
    "DecisionTree": globals().get("best_dt", None),
}

for key, obj in candidates.items():
    if obj is not None and key.lower() in model_name.lower():
        best_model_obj = obj
        break

# если не нашли по имени — попробуем явно best_gb (часто он лучший у тебя)
if best_model_obj is None and globals().get("best_gb", None) is not None:
    best_model_obj = globals().get("best_gb")

if best_model_obj is None:
    raise NameError("Не удалось определить объект лучшей модели. Проверь, что best_dt/best_rf/best_gb существуют.")

joblib.dump(best_model_obj, ART_DIR / "best_model.joblib")
print("Saved:", (ART_DIR / "best_model.joblib").resolve())

best_model_meta = {
    "best_model_name": model_name,
    "best_model_class": best_model_obj.__class__.__name__,
    "best_model_params": best_model_obj.get_params(),
    "selection_criterion": ["roc_auc", "f1", "accuracy"],
    "test_metrics": {
        "accuracy": float(best_row.get("accuracy")),
        "f1": float(best_row.get("f1")),
        "roc_auc": None if pd.isna(best_row.get("roc_auc")) else float(best_row.get("roc_auc")),
    },
    "cv_best_scores": search_summaries.get(best_model_obj.__class__.__name__, None)  # может быть None, это ок
}

(ART_DIR / "best_model_meta.json").write_text(
    json.dumps(best_model_meta, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Saved:", (ART_DIR / "best_model_meta.json").resolve())

# ---------- 4) Проверка, что в figures есть минимум 2 изображения ----------
imgs = sorted([p for p in FIG_DIR.glob("*.png")])
print(f"Figures found: {len(imgs)}")
for p in imgs[:10]:
    print(" -", p.name)


Saved: C:\Users\sveti\PycharmProjects\DPO_II\aie-group\homeworks\HW06\artifacts\metrics_test.json
Saved: C:\Users\sveti\PycharmProjects\DPO_II\aie-group\homeworks\HW06\artifacts\search_summaries.json
Saved: C:\Users\sveti\PycharmProjects\DPO_II\aie-group\homeworks\HW06\artifacts\best_model.joblib
Saved: C:\Users\sveti\PycharmProjects\DPO_II\aie-group\homeworks\HW06\artifacts\best_model_meta.json
Figures found: 6
 - decisiontree_confusion_matrix.png
 - decisiontree_roc_curve.png
 - gradientboosting_confusion_matrix.png
 - gradientboosting_roc_curve.png
 - randomforest_confusion_matrix.png
 - randomforest_roc_curve.png
